<a href="https://colab.research.google.com/github/abhinavreddyjitta/ML_LAB_2026-27/blob/main/Decision_Tree_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = next(iter(uploaded))
data = pd.read_csv(filename)

print(data.head())
print("Dataset shape:", data.shape)

Saving data.csv to data.csv
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_w

In [2]:

print(data.head())
print(data.shape)
print(data.columns)
print(data.isnull().sum())

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

In [3]:

# Remove unnecessary columns
data = data.drop(columns=["id", "Unnamed: 32"], errors="ignore")

# Separate input features and target
X = data.drop(columns=["diagnosis"])
y = data["diagnosis"].map({"M": 1, "B": 0})

# Check the data
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts())

Features shape: (569, 30)
Target shape: (569,)
diagnosis
0    357
1    212
Name: count, dtype: int64


In [4]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [5]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)

print("Decision Tree Accuracy:",
      round(accuracy_score(y_test, y_pred), 4))

Decision Tree Accuracy: 0.9298


In [6]:

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Define the parameters to test
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Create the Decision Tree model
dt = DecisionTreeClassifier(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)

# Train the model using different parameter combinations
grid_search.fit(X_train, y_train)

# Display the best parameters
print("Best Parameters:")
print(grid_search.best_params_)

# Get the best model
best_dt = grid_search.best_estimator_

# Predict using the tuned model
y_pred_tuned = best_dt.predict(X_test)

# Calculate accuracy
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print("\nBest Cross-Validation Accuracy:",
      round(grid_search.best_score_, 4))

print("Tuned Decision Tree Accuracy:",
      round(accuracy_tuned, 4))

Fitting 5 folds for each of 90 candidates, totalling 450 fits
Best Parameters:
{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5}

Best Cross-Validation Accuracy: 0.9473
Tuned Decision Tree Accuracy: 0.9561


In [9]:
from sklearn.metrics import accuracy_score

# Calculate the default decision tree accuracy
accuracy_default = accuracy_score(y_test, y_pred)

print("Default Decision Tree Accuracy:",
      round(accuracy_default, 4))

print("Tuned Decision Tree Accuracy:",
      round(accuracy_tuned, 4))

Default Decision Tree Accuracy: 0.9298
Tuned Decision Tree Accuracy: 0.9561


In [10]:

from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=["Benign", "Malignant"]
))

Confusion Matrix:
[[72  0]
 [ 5 37]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      1.00      0.97        72
   Malignant       1.00      0.88      0.94        42

    accuracy                           0.96       114
   macro avg       0.97      0.94      0.95       114
weighted avg       0.96      0.96      0.96       114

